In [1]:
# 01. Business Problem & Project Overview
USE_CASE = "07 - Production Order Delay Prediction (Predict Delayed Production Orders)"
ML_PARADIGM = "Unsupervised Machine Learning (5 Models Evaluated & Ranked -> #1 Champion Model Selected)"
SCHEMAS = "HANA_P2 and HANA_P3 (All Combined Production Order Data - NO LIMIT)"
N_CLUSTERS = 4

print(f"Business Problem: {USE_CASE}")
print(f"ML Approach:      {ML_PARADIGM}")
print(f"Database Schemas: {SCHEMAS}")
print(f"Delay Risk Tiers: {N_CLUSTERS} Tiers (On-Time, Low Risk, Moderate Risk, Critical Delay Exception)")
print("Objective:        Predict delayed production orders and estimate calibrated Probability of Delay P(Delay)")


Business Problem: 07 - Production Order Delay Prediction (Predict Delayed Production Orders)
ML Approach:      Unsupervised Machine Learning (5 Models Evaluated & Ranked -> #1 Champion Model Selected)
Database Schemas: HANA_P2 and HANA_P3 (All Combined Production Order Data - NO LIMIT)
Delay Risk Tiers: 4 Tiers (On-Time, Low Risk, Moderate Risk, Critical Delay Exception)
Objective:        Predict delayed production orders and estimate calibrated Probability of Delay P(Delay)


In [2]:
# 02. Import Libraries & Set Global Environments
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.cluster import KMeans, MiniBatchKMeans, BisectingKMeans
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    average_precision_score, confusion_matrix, roc_curve, precision_recall_curve,
    silhouette_score
)
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings('ignore')
%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 5)
print("All core ML, clustering, and data processing packages successfully imported.")


All core ML, clustering, and data processing packages successfully imported.


In [3]:
# 03. Database Server Connection & Configuration
from src.db_connector import load_config, get_db_connection

config = load_config()
db = config["database"]
print(f"Database Server: {db['host']}:{db['port']}")
print(f"Database Name:   {db['dbname']}")
print(f"Schemas:         {db.get('schemas', ['HANA_P2', 'HANA_P3'])}")

conn = get_db_connection(config)
print("Live PostgreSQL connection established successfully.")
conn.close()


Database Server: 192.168.1.44:5432
Database Name:   JKC_Input
Schemas:         ['HANA_P2', 'HANA_P3']
Live PostgreSQL connection established successfully.


In [4]:
# 04. Extract All Combined Production Order Data (NO LIMIT)
from src.db_connector import extract_production_order_data

raw_df = extract_production_order_data(limit=None, config=config, use_cache=True)
print(f"Extracted Production Order DataFrame Shape: {raw_df.shape}")
print(f"Total Columns: {len(raw_df.columns)}")


[CACHE] Loading cached full production order dataset from: data/cache/production_order_delay_raw_data_all.csv
[CACHE] Successfully loaded 302,662 records from cache.
Extracted Production Order DataFrame Shape: (302662, 44)
Total Columns: 44


In [5]:
# 05. Exploratory Data Analysis & Schema Inspection
print("Dataset Summary:")
print(raw_df[['order_number', 'order_type', 'plant', 'planned_start_date', 'planned_finish_date', 'technical_completion_date', 'total_order_qty']].head(5))
print("\nTop Order Types:")
print(raw_df['order_type'].value_counts().head(8))


Dataset Summary:
   order_number order_type  ... technical_completion_date total_order_qty
0      10021778       PWR1  ...                2026-06-01      11200000.0
1      10020131       PWR1  ...                2026-05-01      11200000.0
2      10018766       PWR1  ...                2026-04-01      11200000.0
3      10018037       PWR1  ...                2026-03-01      11000000.0
4      10003910       PWR1  ...                2024-11-01      11000000.0

[5 rows x 7 columns]

Top Order Types:
order_type
PM02    165518
PM03     64394
PM07     11769
PM05     11481
PCM1      8782
PM10      8192
BCM1      6247
FG01      5327
Name: count, dtype: int64


In [6]:
# 06. Data Cleaning & 30-Feature Engineering with Missing Value Handling
from src.preprocessing import clean_and_engineer_features, FEATURE_COLUMNS

featured_df = clean_and_engineer_features(raw_df)
print(f"Engineered Features DataFrame Shape: {featured_df.shape}")
print(f"Total Model Features: {len(FEATURE_COLUMNS)}")
print("Missing values in raw attributes are preserved as clean blanks.")


Engineered Features DataFrame Shape: (302662, 90)
Total Model Features: 30
Missing values in raw attributes are preserved as clean blanks.


In [7]:
# 07. Temporal Delay Benchmark & Ground Truth Definition
delay_cnt = int(featured_df['is_delayed_exception'].sum())
delay_rate = (delay_cnt / len(featured_df)) * 100.0
print(f"Total Processed Production Orders: {len(featured_df):,}")
print(f"Ground-Truth Delayed Orders:       {delay_cnt:,} ({delay_rate:.2f}%)")
print(f"Ground-Truth On-Time Orders:       {len(featured_df) - delay_cnt:,} ({100.0 - delay_rate:.2f}%)")


Total Processed Production Orders: 302,662
Ground-Truth Delayed Orders:       266,062 (87.91%)
Ground-Truth On-Time Orders:       36,600 (12.09%)


In [8]:
# 08. Feature Matrix (X) Definition & RobustScaler Partitioning
from src.preprocessing import prepare_train_test_data

data_dict = prepare_train_test_data(featured_df, test_size=0.20, random_state=42)
X_train_scaled = data_dict['X_train_scaled']
X_test_scaled = data_dict['X_test_scaled']
y_train = data_dict['y_train']
y_test = data_dict['y_test']
scaler = data_dict['scaler']

print(f"Training Scaled Matrix X_train: {X_train_scaled.shape}")
print(f"Testing Scaled Matrix X_test:   {X_test_scaled.shape}")
print("Training is strictly UNSUPERVISED (y_train is NOT used in model fitting).")


Training Scaled Matrix X_train: (242129, 30)
Testing Scaled Matrix X_test:   (60533, 30)
Training is strictly UNSUPERVISED (y_train is NOT used in model fitting).


In [9]:
# 09. Instantiate Unsupervised Clustering Suite
from src.models import ProductionOrderDelayClusteringSuite

contamination = max(0.10, min(0.60, delay_cnt / len(featured_df)))
suite = ProductionOrderDelayClusteringSuite(n_clusters=4, contamination=contamination, random_state=42)
print(f"Clustering Suite initialized with n_clusters=4, contamination={contamination:.4f}")


Clustering Suite initialized with n_clusters=4, contamination=0.6000


In [10]:
# 10. Model 1: K-Means Clustering (Centroid-Based Partitioning)
print("Training Model 1/5: K-Means Clustering...")
km = KMeans(n_clusters=4, init='k-means++', n_init=5, random_state=42)
km_labels = km.fit_predict(X_train_scaled)
suite._register_cluster_model('K-Means Clustering', km, X_train_scaled, km_labels)
print("K-Means training and cluster ordering complete.")


Training Model 1/5: K-Means Clustering...
K-Means training and cluster ordering complete.


In [11]:
# 11. Model 2: Gaussian Mixture Model (Probabilistic Density Estimation)
print("Training Model 2/5: Gaussian Mixture Model...")
gmm = GaussianMixture(n_components=4, covariance_type='diag', random_state=42, max_iter=30)
gmm_labels = gmm.fit_predict(X_train_scaled)
suite._register_cluster_model('Gaussian Mixture Model', gmm, X_train_scaled, gmm_labels)
print("Gaussian Mixture Model training and cluster ordering complete.")


Training Model 2/5: Gaussian Mixture Model...
Gaussian Mixture Model training and cluster ordering complete.


In [12]:
# 12. Model 3: Mini-Batch K-Means (Stochastic Partitioning)
print("Training Model 3/5: Mini-Batch K-Means...")
mbk = MiniBatchKMeans(n_clusters=4, batch_size=4096, n_init=5, random_state=42)
mbk_labels = mbk.fit_predict(X_train_scaled)
suite._register_cluster_model('Mini-Batch K-Means', mbk, X_train_scaled, mbk_labels)
print("Mini-Batch K-Means training complete.")


Training Model 3/5: Mini-Batch K-Means...
Mini-Batch K-Means training complete.


In [13]:
# 13. Model 4: Bisecting K-Means (Hierarchical Partitioning)
print("Training Model 4/5: Bisecting K-Means...")
bkm = BisectingKMeans(n_clusters=4, init='k-means++', random_state=42)
bkm_labels = bkm.fit_predict(X_train_scaled)
suite._register_cluster_model('Bisecting K-Means', bkm, X_train_scaled, bkm_labels)
print("Bisecting K-Means training complete.")


Training Model 4/5: Bisecting K-Means...
Bisecting K-Means training complete.


In [14]:
# 14. Model 5: PCA Latent Space Clustering (8-Component Partitioning)
from src.models import PCALatentClusterer

print("Training Model 5/5: PCA Latent Space Clustering...")
pca_km = PCALatentClusterer(n_clusters=4, n_components=8, random_state=42)
pca_km_labels = pca_km.fit_predict(X_train_scaled)
suite._register_cluster_model('PCA Latent Clustering', pca_km, X_train_scaled, pca_km_labels)
print("PCA Latent Space Clustering training complete.")


Training Model 5/5: PCA Latent Space Clustering...
PCA Latent Space Clustering training complete.


In [15]:
# 15. Generate Test Set Predictions & Probabilities
test_preds = suite.predict_all(X_test_scaled)
print(f"Generated predictions and continuous P(Delay) scores across all {len(test_preds)} models.")


Generated predictions and continuous P(Delay) scores across all 5 models.


In [16]:
# 16. Model Comparison Table & Ranking Selection
from src.evaluate import evaluate_models

comparison_df = evaluate_models(test_preds, y_test)
print("=" * 96)
print("5 UNSUPERVISED ML MODELS EVALUATION & RANKING TABLE")
print("=" * 96)
print(comparison_df.to_string(index=False))
print("=" * 96)

champion_model_name = comparison_df.iloc[0]['Model']
print(f"\nCHAMPION MODEL SELECTED: {champion_model_name}")


5 UNSUPERVISED ML MODELS EVALUATION & RANKING TABLE
Rank                  Model  Accuracy  Precision  Recall  F1-Score  ROC-AUC  PR-AUC  Silhouette  Specificity  True Positives  False Positives  False Negatives  True Negatives
  #1  PCA Latent Clustering    0.8514     0.8914  0.9462    0.9180   0.7355  0.9422      0.8010       0.1623           50349             6132             2864            1188
  #2     Mini-Batch K-Means    0.8511     0.8916  0.9455    0.9178   0.7356  0.9422      0.8015       0.1646           50312             6115             2901            1205
  #3     K-Means Clustering    0.8510     0.8916  0.9454    0.9177   0.7356  0.9422      0.8006       0.1648           50309             6114             2904            1206
  #4      Bisecting K-Means    0.8499     0.8931  0.9420    0.9169   0.7352  0.9422      0.7885       0.1807           50124             5997             3089            1323
  #5 Gaussian Mixture Model    0.8267     0.9339  0.8640    0.8976   0.75

In [17]:
# 17. Performance Comparison Visualizations
from src.evaluate import plot_model_comparison

os.makedirs("reports/figures", exist_ok=True)
plot_model_comparison(comparison_df, output_dir="reports/figures")
plt.show()


[REPORT] Saved model comparison chart to reports/figures\model_comparison.png


In [18]:
# 18. Receiver Operating Characteristic & Precision-Recall Curves
from src.evaluate import plot_roc_pr_curves

plot_roc_pr_curves(test_preds, y_test, output_dir="reports/figures")
plt.show()


[REPORT] Saved ROC & PR curves to reports/figures\roc_pr_curves.png


In [19]:
# 19. Deploy #1 Champion Model & Production Scoring
from src.models import SingleModelProductionOrderSegmenter
from src.inference import score_with_single_model

champion_segmenter = SingleModelProductionOrderSegmenter(
    model_name=champion_model_name,
    n_clusters=4,
    contamination=contamination,
    random_state=42
)
champion_segmenter.fit(X_train_scaled)

scored_df = score_with_single_model(champion_segmenter, featured_df, scaler, FEATURE_COLUMNS)
print("Production scoring complete. Scored columns preview:")
print(scored_df[['order_number', 'order_type', 'plant', 'predicted_delay_probability', 'delay_prediction_label', 'delay_risk_tier', 'primary_delay_driver']].head(5))


Production scoring complete. Scored columns preview:
  order_number  ...                               primary_delay_driver
0     10021778  ...       Severe Fulfillment Backlog (35.4% delivered)
1     10020131  ...  Cumulative Lead Time Strain & Operational Bott...
2     10018766  ...  Cumulative Lead Time Strain & Operational Bott...
3     10018037  ...  Cumulative Lead Time Strain & Operational Bott...
4     10003910  ...       Severe Fulfillment Backlog (42.6% delivered)

[5 rows x 7 columns]


In [20]:
# 20. Feature Importance & Operational Delay Drivers
from src.evaluate import compute_feature_importance, plot_feature_importance, plot_delay_risk_distribution

fi_df = compute_feature_importance(X_train_scaled, y_train, FEATURE_COLUMNS)
plot_feature_importance(fi_df, output_dir="reports/figures", top_n=15)
plt.show()

plot_delay_risk_distribution(scored_df, output_dir="reports/figures")
plt.show()

print("Top 10 Delay Drivers:")
print(fi_df.head(10)[['Rank', 'Feature', 'Importance', 'Relative_Weight_Pct']].to_string(index=False))


[REPORT] Saved feature importance plot to reports/figures\feature_importance.png
[REPORT] Saved delay risk distribution plot to reports/figures\delay_risk_distribution.png
Top 10 Delay Drivers:
 Rank                       Feature  Importance  Relative_Weight_Pct
    1    confirmation_duration_days    0.134557                13.46
    2        composite_delay_signal    0.118712                11.87
    3         planned_duration_days    0.107004                10.70
    4    schedule_compression_index    0.067929                 6.79
    5            release_delay_days    0.065897                 6.59
    6           unfulfilled_qty_log    0.057349                 5.73
    7        confirmation_intensity    0.056129                 5.61
    8           delivered_qty_ratio    0.049327                 4.93
    9 scheduled_vs_basic_slack_days    0.048275                 4.83
   10                 plant_encoded    0.038901                 3.89


In [21]:
# 21. Multi-Sheet Executive Excel Export & Model Persistence
import shutil
from src.inference import export_enriched_excel_report, save_single_model_pipeline

output_excel = "ML_7_Production_Order_Delay_Prediction_Enriched_Output_Clean.xlsx"
export_enriched_excel_report(scored_df, output_path=output_excel, fi_df=fi_df, comp_df=comparison_df)

alt_excel = "ML_7_Production_Order_Delay_Prediction_Enriched_Output.xlsx"
try:
    shutil.copyfile(output_excel, alt_excel)
    print(f"Copied report to {alt_excel}")
except PermissionError:
    print(f"Notice: {alt_excel} is currently open in Excel. Updated file is at {output_excel}.")

save_single_model_pipeline(champion_segmenter, scaler, FEATURE_COLUMNS, output_dir="models")
print("All deliverables generated successfully!")


[REPORT] Generating multi-sheet executive Excel workbook with xlsxwriter: ML_7_Production_Order_Delay_Prediction_Enriched_Output_Clean.xlsx...
[REPORT] Successfully generated and closed Excel workbook: ML_7_Production_Order_Delay_Prediction_Enriched_Output_Clean.xlsx
Copied report to ML_7_Production_Order_Delay_Prediction_Enriched_Output.xlsx
[MODEL] Champion Model persisted to: models\champion_delay_model.joblib
[MODEL] Preprocessing Scaler persisted to: models\delay_scaler.joblib
[MODEL] Complete Deployment Bundle persisted to: models\delay_model_bundle.joblib
All deliverables generated successfully!
